In [1]:
# Install dependencies for this notebook kernel
%pip install --break-system-packages --upgrade pandas matplotlib seaborn wordcloud

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached wordcloud-1.9.6-cp314-cp314-win_amd64.whl.metadata (3.5 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached matplotlib-3.10.9-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached seab

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Spam Classifier - Exploratory Data Analysis
This notebook explores the South African spam dataset and provides insights into the data characteristics.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load the Data

In [ ]:
# Load raw data
df = pd.read_csv('../data/raw/spam.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Dataset Overview

In [ ]:
# Basic info
df.info()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

## 3. Label Distribution

In [ ]:
# Label counts
label_counts = df['label'].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nSpam percentage: {label_counts['spam']/len(df)*100:.2f}%")
print(f"Ham percentage: {label_counts['ham']/len(df)*100:.2f}%")

In [ ]:
# Visualize label distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
label_counts.plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Email Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Label', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_xticklabels(['Ham', 'Spam'], rotation=0)

# Pie chart
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', 
                  colors=['#2ecc71', '#e74c3c'], startangle=90)
ax2.set_title('Email Distribution (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Text Length Analysis

In [ ]:
# Combine subject and body
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')

# Calculate text lengths
df['text_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# Summary statistics
print("Text length statistics by label:")
print(df.groupby('label')[['text_length', 'word_count']].describe())

In [ ]:
# Visualize text length distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Character length
df[df['label']=='ham']['text_length'].hist(bins=50, alpha=0.6, label='Ham', ax=ax1, color='#2ecc71')
df[df['label']=='spam']['text_length'].hist(bins=50, alpha=0.6, label='Spam', ax=ax1, color='#e74c3c')
ax1.set_title('Text Length Distribution (Characters)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Character Count', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.legend()

# Word count
df[df['label']=='ham']['word_count'].hist(bins=50, alpha=0.6, label='Ham', ax=ax2, color='#2ecc71')
df[df['label']=='spam']['word_count'].hist(bins=50, alpha=0.6, label='Spam', ax=ax2, color='#e74c3c')
ax2.set_title('Word Count Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Word Count', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Common Words Analysis

In [ ]:
def get_top_words(texts, n=20):
    """Extract top N most common words from a collection of texts."""
    all_words = []
    for text in texts:
        words = re.findall(r'\b[a-z]{3,}\b', str(text).lower())
        all_words.extend(words)
    
    word_counts = Counter(all_words)
    return word_counts.most_common(n)

# Get top words for spam and ham
spam_words = get_top_words(df[df['label']=='spam']['text'])
ham_words = get_top_words(df[df['label']=='ham']['text'])

print("Top 20 words in SPAM messages:")
for word, count in spam_words:
    print(f"{word:15s}: {count:5d}")

print("\nTop 20 words in HAM messages:")
for word, count in ham_words:
    print(f"{word:15s}: {count:5d}")

In [ ]:
# Visualize top words
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Spam words
words, counts = zip(*spam_words)
ax1.barh(words, counts, color='#e74c3c')
ax1.set_title('Top 20 Words in SPAM Messages', fontsize=14, fontweight='bold')
ax1.set_xlabel('Frequency', fontsize=12)
ax1.invert_yaxis()

# Ham words
words, counts = zip(*ham_words)
ax2.barh(words, counts, color='#2ecc71')
ax2.set_title('Top 20 Words in HAM Messages', fontsize=14, fontweight='bold')
ax2.set_xlabel('Frequency', fontsize=12)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Word Clouds

In [ ]:
# Create word clouds
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Spam word cloud
spam_text = ' '.join(df[df['label']=='spam']['text'].astype(str))
spam_wordcloud = WordCloud(width=800, height=400, 
                           background_color='white',
                           colormap='Reds').generate(spam_text)
ax1.imshow(spam_wordcloud, interpolation='bilinear')
ax1.set_title('SPAM Messages Word Cloud', fontsize=16, fontweight='bold')
ax1.axis('off')

# Ham word cloud
ham_text = ' '.join(df[df['label']=='ham']['text'].astype(str))
ham_wordcloud = WordCloud(width=800, height=400, 
                          background_color='white',
                          colormap='Greens').generate(ham_text)
ax2.imshow(ham_wordcloud, interpolation='bilinear')
ax2.set_title('HAM Messages Word Cloud', fontsize=16, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 7. Special Pattern Analysis

In [ ]:
# Analyze special patterns
df['has_url'] = df['text'].str.contains(r'http|www', case=False, na=False)
df['has_phone'] = df['text'].str.contains(r'\d{3}\s?\d{3}\s?\d{4}|\+27', case=False, na=False)
df['has_money'] = df['text'].str.contains(r'\br\s?\d+', case=False, na=False)
df['has_caps'] = df['text'].str.contains(r'[A-Z]{3,}', na=False)
df['has_exclamation'] = df['text'].str.contains(r'!!!', na=False)

# Calculate percentages by label
patterns = ['has_url', 'has_phone', 'has_money', 'has_caps', 'has_exclamation']
pattern_analysis = pd.DataFrame()

for pattern in patterns:
    spam_pct = df[df['label']=='spam'][pattern].mean() * 100
    ham_pct = df[df['label']=='ham'][pattern].mean() * 100
    pattern_analysis[pattern] = [spam_pct, ham_pct]

pattern_analysis.index = ['Spam', 'Ham']
pattern_analysis.columns = ['URLs', 'Phone Numbers', 'Money Amounts', 'CAPITAL WORDS', 'Multiple !!!']

print("Special Pattern Analysis (% of messages containing pattern):")
print(pattern_analysis.round(2))

In [ ]:
# Visualize pattern analysis
pattern_analysis.T.plot(kind='bar', figsize=(12, 6), color=['#e74c3c', '#2ecc71'])
plt.title('Special Patterns in Spam vs Ham Messages', fontsize=14, fontweight='bold')
plt.xlabel('Pattern Type', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(['Spam', 'Ham'])
plt.tight_layout()
plt.show()

## 8. Sample Messages

In [ ]:
# Show sample spam messages
print("Sample SPAM messages:")
print("=" * 80)
for i, row in df[df['label']=='spam'].sample(5, random_state=42).iterrows():
    print(f"Subject: {row['subject']}")
    print(f"Body: {row['body'][:100]}...")
    print("-" * 80)

In [ ]:
# Show sample ham messages
print("Sample HAM messages:")
print("=" * 80)
for i, row in df[df['label']=='ham'].sample(5, random_state=42).iterrows():
    print(f"Subject: {row['subject']}")
    print(f"Body: {row['body'][:100]}...")
    print("-" * 80)

## Conclusions

Key findings from the exploratory analysis:

1. **Class Balance**: The dataset shows class distribution between spam and ham messages
2. **Text Length**: Spam messages tend to have different length characteristics than ham
3. **Common Patterns**: Spam messages frequently contain:
   - Money amounts (R values)
   - Phone numbers
   - Multiple exclamation marks
   - Capital letters
   - Urgency words ("NOW", "GUARANTEED", "FREE")
4. **South African Context**: The dataset includes SA-specific spam patterns:
   - Sangoma/traditional healer scams
   - Forex trading schemes
   - Work-from-home opportunities
   - Government grant scams
   - Cash loan offers

These insights will help in feature engineering and model selection.